In [1]:
import json
import sys
from typing import List, Dict
from config import*
from unittest.mock import Mock, patch
import pandas as pd
from utils.utils import dump
from models.project_model import UseCase
from models.project_model import *
import unicodedata
from google import genai

In [2]:
SAMPLE_USECASES_TEXT = """
Cas d'utilisation 1: Authentification utilisateur
- L'utilisateur saisit ses identifiants
- Le système vérifie les informations
- L'utilisateur accède au tableau de bord

Cas d'utilisation 2: Gestion des commandes
- L'utilisateur consulte le catalogue
- L'utilisateur ajoute des articles au panier
- L'utilisateur finalise la commande

Cas d'utilisation 3: Validation de commande
- Le système vérifie la disponibilité des articles
- Le système calcule le montant total
- Le système confirme la commande
"""

SAMPLE_USECASE_JSON = [
    {
        "name": "Authentification utilisateur",
        "action": "POST",
        "actors": ["Utilisateur", "Système"],
        "scenarios": {
            "principal": [
                "L'utilisateur saisit ses identifiants",
                "Le système vérifie les informations", 
                "L'utilisateur accède au tableau de bord"
            ],
            "alternatif": [
                "Identifiants incorrects",
                "Le système affiche un message d'erreur"
            ]
        },
        "preconditions": ["L'utilisateur possède un compte"],
        "postconditions": ["L'utilisateur est connecté"],
        "uses": "",
        "extends": ""
    },
    {
        "name": "Gestion des commandes", 
        "action": "POST",
        "actors": ["Utilisateur"],
        "scenarios": {
            "principal": [
                "L'utilisateur consulte le catalogue",
                "L'utilisateur ajoute des articles au panier",
                "L'utilisateur finalise la commande"
            ],
            "alternatif": [
                "Articles indisponibles",
                "Le système propose des alternatives"
            ]
        },
        "preconditions": ["L'utilisateur est authentifié"],
        "postconditions": ["La commande est enregistrée"],
        "uses": "Authentification utilisateur",
        "extends": ""
    }
]

print("✅ Données de test préparées")


✅ Données de test préparées


In [3]:
def determine_http_action(usecase_data: Dict) -> str:
    """Version de test de determine_http_action"""
    name = usecase_data.get('name', '').lower()
    scenarios = usecase_data.get('scenarios', {})
    
    principal_scenario = scenarios.get('principal', [])
    if isinstance(principal_scenario, list):
        principal_text = ' '.join(principal_scenario).lower()
    else:
        principal_text = str(principal_scenario).lower()
    
    create_keywords = ['crée', 'créer', 'ajouter', 'nouveau', 'enregistrer', 'sauvegarder', 'create', 'add', 'insert', 'register', 'submit']
    update_keywords = ['modifie', 'modifier', 'mettre à jour', 'éditer', 'changer', 'update', 'modify', 'edit', 'change']
    delete_keywords = ['supprimer', 'supprime', 'effacer', 'retirer', 'delete', 'remove', 'cancel']
    
    if any(keyword in name or keyword in principal_text for keyword in create_keywords):
        return 'POST'
    elif any(keyword in name or keyword in principal_text for keyword in update_keywords):
        return 'PUT'
    elif any(keyword in name or keyword in principal_text for keyword in delete_keywords):
        return 'DELETE'
    else:
        return 'GET'

def extract_actors(actors_data) -> List[str]:
    """Version de test de extract_actors"""
    if isinstance(actors_data, list):
        return [actor.strip() for actor in actors_data if actor.strip()]
    elif isinstance(actors_data, str):
        return [actor.strip() for actor in actors_data.split(',') if actor.strip()]
    return []

def extract_scenarios(scenarios_data: Dict) -> Scenario:
    """Version de test de extract_scenarios"""
    main = []
    alternative = []
    
    if isinstance(scenarios_data, dict):
        main = scenarios_data.get('principal', [])
        alternative = scenarios_data.get('alternatif', [])
        
        if not main:
            main = scenarios_data.get('main', [])
        if not alternative:
            alternative = scenarios_data.get('alternative', [])
    
    if isinstance(main, str):
        main = [main]
    if isinstance(alternative, str):
        alternative = [alternative]
        
    return Scenario(main=main, alternative=alternative)

def remove_accents(input_str: str) -> str:
    nfkd_form = unicodedata.normalize('NFKD', input_str)
    only_ascii = "".join([c for c in nfkd_form if not unicodedata.combining(c)])
    return only_ascii

def capitalize(text: str) -> str:
    """Version de test de capitalize"""
    return ''.join(word.capitalize() for word in remove_accents(text).replace('-', ' ').replace('_', ' ').split())

In [4]:
def test_utility_functions():
    print("🧪 Test des fonctions utilitaires")
    
    # Test determine_http_action
    test_cases = [
        ({"name": "Créer utilisateur", "scenarios": {"principal": ["ajouter un utilisateur"]}}, "POST"),
        ({"name": "Modifier profil", "scenarios": {"principal": ["mettre à jour les informations"]}}, "PUT"),
        ({"name": "Supprimer compte", "scenarios": {"principal": ["effacer le compte"]}}, "DELETE"),
        ({"name": "Consulter liste", "scenarios": {"principal": ["afficher les données"]}}, "GET")
    ]
    
    for usecase_data, expected in test_cases:
        result = determine_http_action(usecase_data)
        status = "✅" if result == expected else "❌"
        print(f"  {status} determine_http_action: {usecase_data['name']} -> {result} (attendu: {expected})")
    
    # Test extract_actors
    actors_tests = [
        (["User", "Admin"], ["User", "Admin"]),
        ("User, Admin, System", ["User", "Admin", "System"]),
        ([], [])
    ]
    
    for input_data, expected in actors_tests:
        result = extract_actors(input_data)
        status = "✅" if result == expected else "❌"
        print(f"  {status} extract_actors: {input_data} -> {result}")
    
    # Test extract_scenarios
    scenario_data = {"principal": ["étape 1", "étape 2"], "alternatif": ["erreur"]}
    result = extract_scenarios(scenario_data)
    status = "✅" if len(result.main) == 2 and len(result.alternative) == 1 else "❌"
    print(f"  {status} extract_scenarios: scénarios extraits correctement")
    
    # Test capitalize
    capitalize_tests = [
        ("créer utilisateur", "CreerUtilisateur"),
        ("gestion-des-commandes", "GestionDesCommandes"),
        ("test_simple", "TestSimple")
    ]
    
    for input_text, expected in capitalize_tests:
        result = capitalize(input_text)
        status = "✅" if result == expected else "❌"
        print(f"  {status} capitalize: '{input_text}' -> '{result}' (attendu: '{expected}')")


In [5]:
test_utility_functions()

🧪 Test des fonctions utilitaires
  ✅ determine_http_action: Créer utilisateur -> POST (attendu: POST)
  ✅ determine_http_action: Modifier profil -> PUT (attendu: PUT)
  ✅ determine_http_action: Supprimer compte -> DELETE (attendu: DELETE)
  ✅ determine_http_action: Consulter liste -> GET (attendu: GET)
  ✅ extract_actors: ['User', 'Admin'] -> ['User', 'Admin']
  ✅ extract_actors: User, Admin, System -> ['User', 'Admin', 'System']
  ✅ extract_actors: [] -> []
  ✅ extract_scenarios: scénarios extraits correctement
  ✅ capitalize: 'créer utilisateur' -> 'CreerUtilisateur' (attendu: 'CreerUtilisateur')
  ✅ capitalize: 'gestion-des-commandes' -> 'GestionDesCommandes' (attendu: 'GestionDesCommandes')
  ✅ capitalize: 'test_simple' -> 'TestSimple' (attendu: 'TestSimple')


In [6]:
class MockResponse:
    def __init__(self, text):
        self.text = text
    
    def to_dict(self):
        return {"text": self.text}

In [7]:
class MockGeminiClient:
    def __init__(self):
        self.models = self
    
    def generate_content(self, model, contents):
        # Simuler différentes réponses selon le contenu du prompt
        if "attributs appropriés pour un DTO" in contents:
            return MockResponse(json.dumps([
                {
                    "name": "id",
                    "type": "Long", 
                    "visibility": "private",
                    "decorators": [{"name": "NotNull", "message": "ID is required"}]
                },
                {
                    "name": "username",
                    "type": "String",
                    "visibility": "private", 
                    "decorators": [{"name": "NotBlank", "message": "Username cannot be blank"}]
                }
            ]))
        elif "méthodes appropriées pour un Service" in contents:
            return MockResponse(json.dumps([
                {
                    "name": "authenticate",
                    "visibility": "public",
                    "type": "AuthenticationDto",
                    "args": [{"name": "dto", "type": "AuthenticationDto", "visibility": "public"}]
                }
            ]))
        elif "méthodes appropriées pour un Repository" in contents:
            return MockResponse(json.dumps([
                {
                    "name": "findByUsername",
                    "visibility": "public", 
                    "type": "Optional<AuthenticationEntity>",
                    "args": [{"name": "username", "type": "String", "visibility": "public"}]
                },
                {
                    "name": "save",
                    "visibility": "public",
                    "type": "AuthenticationEntity", 
                    "args": [{"name": "entity", "type": "AuthenticationEntity", "visibility": "public"}]
                }
            ]))
        elif "relations sémantiques" in contents:
            return MockResponse(json.dumps({
                "relations_identifiees": [
                    {
                        "nom_cas_utilisation": "Gestion des commandes",
                        "type_relation": "USES",
                        "force_relation": "forte",
                        "justification": "La gestion des commandes nécessite une authentification préalable"
                    }
                ]
            }))
        else:
            return MockResponse('{"result": "mock_response"}')

In [8]:
class MockGeminiClient:
    def __init__(self):
        self.models = self
    
    def generate_content(self, model, contents):
        if "attributs appropriés pour un DTO" in contents:
            return MockResponse(json.dumps([
                {
                    "name": "id",
                    "type": "Long", 
                    "visibility": "private",
                    "decorators": [{"name": "NotNull", "message": "ID is required"}]
                },
                {
                    "name": "username",
                    "type": "String",
                    "visibility": "private", 
                    "decorators": [{"name": "NotBlank", "message": "Username cannot be blank"}]
                }
            ]))
        elif "méthodes appropriées pour un Service" in contents:
            return MockResponse(json.dumps([
                {
                    "name": "authenticate",
                    "visibility": "public",
                    "type": "AuthenticationDto",
                    "args": [{"name": "dto", "type": "AuthenticationDto", "visibility": "public"}]
                }
            ]))
        elif "méthodes appropriées pour un Repository" in contents:
            return MockResponse(json.dumps([
                {
                    "name": "findByUsername",
                    "visibility": "public", 
                    "type": "Optional<AuthenticationEntity>",
                    "args": [{"name": "username", "type": "String", "visibility": "public"}]
                },
                {
                    "name": "save",
                    "visibility": "public",
                    "type": "AuthenticationEntity", 
                    "args": [{"name": "entity", "type": "AuthenticationEntity", "visibility": "public"}]
                }
            ]))
        elif "relations sémantiques" in contents:
            return MockResponse(json.dumps({
                "relations_identifiees": [
                    {
                        "nom_cas_utilisation": "Gestion des commandes",
                        "type_relation": "USES",
                        "force_relation": "forte",
                        "justification": "La gestion des commandes nécessite une authentification préalable"
                    }
                ]
            }))
        else:
            return MockResponse('{"result": "mock_response"}')

In [10]:
def test_gemini_generators():
    print("🧪 Test des générateurs Gemini (mocked)")
    
    # Mock du client Gemini
    with patch('google.genai.Client', return_value=MockGeminiClient()):
        
        # Test génération DTO
        try:
            # Simuler generate_dto_attributes_with_gemini
            mock_attributes = [
                DtoAttribute("private", "id", "Long", [Decorator("NotNull", "ID is required")]),
                DtoAttribute("private", "username", "String", [Decorator("NotBlank", "Username cannot be blank")])
            ]
            print("  ✅ Génération attributs DTO: Mock réussi")
            print(f"    - {len(mock_attributes)} attributs générés")
            for attr in mock_attributes:
                print(f"    - {attr.name}: {attr.type} ({len(attr.decorators)} décorateurs)")
        except Exception as e:
            print(f"  ❌ Génération attributs DTO: {e}")
        
        # Test génération Service
        try:
            mock_methods = [
                Method("authenticate", [Arg("public", "dto", "AuthenticationDto")], "AuthenticationDto", "public")
            ]
            print("  ✅ Génération méthodes Service: Mock réussi")
            print(f"    - {len(mock_methods)} méthodes générées")
            for method in mock_methods:
                print(f"    - {method.name}({len(method.args)} args) -> {method.type}")
        except Exception as e:
            print(f"  ❌ Génération méthodes Service: {e}")
        
        # Test génération Repository
        try:
            mock_repo_methods = [
                Method("findByUsername", [Arg("public", "username", "String")], "Optional<AuthenticationEntity>", "public"),
                Method("save", [Arg("public", "entity", "AuthenticationEntity")], "AuthenticationEntity", "public")
            ]
            print("  ✅ Génération méthodes Repository: Mock réussi")
            print(f"    - {len(mock_repo_methods)} méthodes générées")
            for method in mock_repo_methods:
                print(f"    - {method.name}({len(method.args)} args) -> {method.type}")
        except Exception as e:
            print(f"  ❌ Génération méthodes Repository: {e}")

In [11]:
test_gemini_generators()

🧪 Test des générateurs Gemini (mocked)
  ✅ Génération attributs DTO: Mock réussi
    - 2 attributs générés
    - id: Long (1 décorateurs)
    - username: String (1 décorateurs)
  ✅ Génération méthodes Service: Mock réussi
    - 1 méthodes générées
    - authenticate(1 args) -> AuthenticationDto
  ✅ Génération méthodes Repository: Mock réussi
    - 2 méthodes générées
    - findByUsername(1 args) -> Optional<AuthenticationEntity>
    - save(1 args) -> AuthenticationEntity


In [12]:
def find_referenced_usecases(usecase: UseCase) -> List[UseCase]:
    """
    Version de test de la fonction find_referenced_usecases
    Utilise des mocks pour simuler l'appel à Gemini
    """
    # Simuler l'accès à la liste globale
    all_usecases = globals().get('all_parsed_usecases', [])
    if not all_usecases:
        return []
    
    # Mock de l'analyse Gemini
    referenced = []
    # Logique simplifiée pour les tests
    for other_uc in all_usecases:
        if other_uc.name != usecase.name:
            # Vérifier les acteurs communs
            common_actors = set(usecase.actors) & set(other_uc.actors)
            
            # Vérifier les références dans les préconditions
            usecase_name_words = usecase.name.lower().split()
            other_preconditions = ' '.join(other_uc.preconditions).lower()
            
            if (len(common_actors) > 0 or 
                any(word in other_preconditions for word in usecase_name_words if len(word) > 3)):
                referenced.append(other_uc)
    
    return referenced

In [16]:
def test_find_referenced_usecases():
    print("🧪 Test de find_referenced_usecases")
    
    # Créer des objets de test pour les composants manquants
    
    # DTOs de test
    dto1 = Dto(
        name="AuthenticationDto",
        attributes=[
            DtoAttribute(
                visibility="private",
                name="username",
                type="String",
                decorators=[Decorator(name="NotNull", message="Username is required")]
            ),
            DtoAttribute(
                visibility="private", 
                name="password",
                type="String",
                decorators=[Decorator(name="NotNull", message="Password is required")]
            )
        ]
    )
    
    dto2 = Dto(
        name="OrderDto",
        attributes=[
            DtoAttribute(
                visibility="private",
                name="id",
                type="Long",
                decorators=[Decorator(name="NotNull", message="ID is required")]
            ),
            DtoAttribute(
                visibility="private",
                name="items",
                type="List<String>",
                decorators=[Decorator(name="NotEmpty", message="Items list cannot be empty")]
            )
        ]
    )
    
    dto3 = Dto(
        name="CatalogDto", 
        attributes=[
            DtoAttribute(
                visibility="private",
                name="productId",
                type="Long",
                decorators=[Decorator(name="NotNull", message="Product ID is required")]
            ),
            DtoAttribute(
                visibility="private",
                name="name",
                type="String", 
                decorators=[Decorator(name="NotBlank", message="Product name is required")]
            )
        ]
    )
    
    # Services de test
    service1 = Service(
        name="AuthenticationService",
        methods=[
            Method(
                name="authenticate",
                args=[Arg("public", "dto", "AuthenticationDto")],
                type="AuthenticationDto",
                visibility="public"
            )
        ]
    )
    
    service2 = Service(
        name="OrderService", 
        methods=[
            Method(
                name="createOrder",
                args=[Arg("public", "dto", "OrderDto")],
                type="OrderDto",
                visibility="public"
            )
        ]
    )
    
    service3 = Service(
        name="CatalogService",
        methods=[
            Method(
                name="getCatalog",
                args=[],
                type="List<CatalogDto>",
                visibility="public"
            )
        ]
    )
    
    # Resources de test
    resource1 = Resource(
        name="AuthenticationResource",
        attributes=[
            Arg("private", "username", "String"),
            Arg("private", "password", "String")
        ]
    )
    
    resource2 = Resource(
        name="OrderResource",
        attributes=[
            Arg("private", "id", "Long"),
            Arg("private", "items", "List<String>")
        ]
    )
    
    resource3 = Resource(
        name="CatalogResource",
        attributes=[
            Arg("private", "productId", "Long"),
            Arg("private", "name", "String")
        ]
    )
    
    # Repositories de test
    repository1 = Repository(
        entity="UserEntity",
        methods=[
            Method(
                name="findByUsername",
                args=[Arg("public", "username", "String")],
                type="Optional<UserEntity>",
                visibility="public"
            ),
            Method(
                name="save",
                args=[Arg("public", "entity", "UserEntity")],
                type="UserEntity", 
                visibility="public"
            )
        ]
    )
    
    repository2 = Repository(
        entity="OrderEntity",
        methods=[
            Method(
                name="save",
                args=[Arg("public", "entity", "OrderEntity")],
                type="OrderEntity",
                visibility="public"
            ),
            Method(
                name="findById",
                args=[Arg("public", "id", "Long")],
                type="Optional<OrderEntity>",
                visibility="public"
            )
        ]
    )
    
    repository3 = Repository(
        entity="ProductEntity",
        methods=[
            Method(
                name="findAll",
                args=[],
                type="List<ProductEntity>",
                visibility="public"
            ),
            Method(
                name="findById",
                args=[Arg("public", "id", "Long")],
                type="Optional<ProductEntity>",
                visibility="public"
            )
        ]
    )
    
    # Créer des cas d'utilisation complets
    usecase1 = UseCase(
        name="Authentification utilisateur",
        action="POST",
        actors=["Utilisateur", "Système"],
        scenarios=Scenario(
            main=["L'utilisateur saisit ses identifiants", "Le système vérifie"],
            alternative=["Identifiants incorrects"]
        ),
        preconditions=["L'utilisateur possède un compte"],
        postconditions=["L'utilisateur est connecté"],
        dto=dto1,
        uses=[],  # Sera rempli par find_referenced_usecases
        extends=[],
        include=[],
        services=[service1],
        resource=resource1,
        repositories=[repository1]
    )
    
    usecase2 = UseCase(
        name="Gestion des commandes",
        action="POST", 
        actors=["Utilisateur"],
        scenarios=Scenario(
            main=["L'utilisateur consulte le catalogue", "L'utilisateur finalise"],
            alternative=["Articles indisponibles"]
        ),
        preconditions=["L'utilisateur est authentifié", "authentification utilisateur requis"],
        postconditions=["La commande est enregistrée"],
        dto=dto2,
        uses=[],  # Sera rempli par find_referenced_usecases
        extends=[],
        include=[],
        services=[service2],
        resource=resource2,
        repositories=[repository2]
    )
    
    usecase3 = UseCase(
        name="Consultation catalogue",
        action="GET",
        actors=["Utilisateur", "Invité"],
        scenarios=Scenario(
            main=["L'utilisateur parcourt les articles"],
            alternative=["Aucun article disponible"]
        ),
        preconditions=["Le catalogue est disponible"],
        postconditions=["Les articles sont affichés"],
        dto=dto3,
        uses=[],  # Sera rempli par find_referenced_usecases
        extends=[],
        include=[],
        services=[service3],
        resource=resource3,
        repositories=[repository3]
    )
    
    # Définir la liste globale pour les tests
    globals()['all_parsed_usecases'] = [usecase1, usecase2, usecase3]
    
    print("📋 Cas d'utilisation créés:")
    print(f"  - {usecase1.name} ({usecase1.action})")
    print(f"  - {usecase2.name} ({usecase2.action})")  
    print(f"  - {usecase3.name} ({usecase3.action})")
    print()
    
    # Test 1: Relations de usecase1
    print("🔍 Test 1: Analyse des relations de 'Authentification utilisateur'")
    try:
        references = find_referenced_usecases(usecase1)
        print(f"  ✅ UseCase1 références: {len(references)} cas d'utilisation trouvés")
        for ref in references:
            print(f"    - {ref.name}")
        if len(references) == 0:
            print("  ℹ️ Aucune relation détectée (normal pour l'authentification)")
    except Exception as e:
        print(f"  ❌ Erreur test UseCase1: {e}")
    print()
    
    # Test 2: Relations de usecase2 (devrait référencer usecase1)
    print("🔍 Test 2: Analyse des relations de 'Gestion des commandes'")
    try:
        references = find_referenced_usecases(usecase2)
        print(f"  ✅ UseCase2 références: {len(references)} cas d'utilisation trouvés")
        for ref in references:
            print(f"    - {ref.name}")
            
        # Vérifier que usecase1 est bien référencé
        referenced_names = [ref.name for ref in references]
        if "Authentification utilisateur" in referenced_names:
            print("    ✅ Relation 'Authentification utilisateur' détectée correctement")
        else:
            print("    ⚠️ Relation 'Authentification utilisateur' non détectée")
            
        if "Consultation catalogue" in referenced_names:
            print("    ✅ Relation 'Consultation catalogue' détectée correctement")
        else:
            print("    ⚠️ Relation 'Consultation catalogue' non détectée")
    except Exception as e:
        print(f"  ❌ Erreur test UseCase2: {e}")
    print()
    
    # Test 3: Relations de usecase3
    print("🔍 Test 3: Analyse des relations de 'Consultation catalogue'")
    try:
        references = find_referenced_usecases(usecase3)
        print(f"  ✅ UseCase3 références: {len(references)} cas d'utilisation trouvés")
        for ref in references:
            print(f"    - {ref.name}")
        if len(references) == 0:
            print("  ℹ️ Aucune relation détectée (normal pour une consultation publique)")
    except Exception as e:
        print(f"  ❌ Erreur test UseCase3: {e}")
    print()
    
    # Test 4: Vérification des relations bidirectionnelles
    print("🔄 Test 4: Vérification des relations bidirectionnelles")
    try:
        # Test si usecase3 est référencé par usecase2
        usecase2_refs = find_referenced_usecases(usecase2)
        usecase3_refs = find_referenced_usecases(usecase3)
        
        usecase2_ref_names = [ref.name for ref in usecase2_refs]
        usecase3_ref_names = [ref.name for ref in usecase3_refs]
        
        if "Consultation catalogue" in usecase2_ref_names:
            print("  ✅ Relation UseCase2 -> UseCase3 détectée")
        if "Gestion des commandes" in usecase3_ref_names:
            print("  ✅ Relation UseCase3 -> UseCase2 détectée")
        
        print(f"  📊 Résumé des relations:")
        print(f"    - UseCase1 référence: {len(find_referenced_usecases(usecase1))} cas")
        print(f"    - UseCase2 référence: {len(find_referenced_usecases(usecase2))} cas")
        print(f"    - UseCase3 référence: {len(find_referenced_usecases(usecase3))} cas")
        
    except Exception as e:
        print(f"  ❌ Erreur test relations bidirectionnelles: {e}")
    
    print("\n🏁 Test terminé!")


# Lancer le test
if __name__ == "__main__":
    test_find_referenced_usecases()

🧪 Test de find_referenced_usecases
📋 Cas d'utilisation créés:
  - Authentification utilisateur (POST)
  - Gestion des commandes (POST)
  - Consultation catalogue (GET)

🔍 Test 1: Analyse des relations de 'Authentification utilisateur'
  ✅ UseCase1 références: 2 cas d'utilisation trouvés
    - Gestion des commandes
    - Consultation catalogue

🔍 Test 2: Analyse des relations de 'Gestion des commandes'
  ✅ UseCase2 références: 2 cas d'utilisation trouvés
    - Authentification utilisateur
    - Consultation catalogue
    ✅ Relation 'Authentification utilisateur' détectée correctement
    ✅ Relation 'Consultation catalogue' détectée correctement

🔍 Test 3: Analyse des relations de 'Consultation catalogue'
  ✅ UseCase3 références: 2 cas d'utilisation trouvés
    - Authentification utilisateur
    - Gestion des commandes

🔄 Test 4: Vérification des relations bidirectionnelles
  ✅ Relation UseCase2 -> UseCase3 détectée
  ✅ Relation UseCase3 -> UseCase2 détectée
  📊 Résumé des relations:
   

In [17]:
test_find_referenced_usecases()

🧪 Test de find_referenced_usecases
📋 Cas d'utilisation créés:
  - Authentification utilisateur (POST)
  - Gestion des commandes (POST)
  - Consultation catalogue (GET)

🔍 Test 1: Analyse des relations de 'Authentification utilisateur'
  ✅ UseCase1 références: 2 cas d'utilisation trouvés
    - Gestion des commandes
    - Consultation catalogue

🔍 Test 2: Analyse des relations de 'Gestion des commandes'
  ✅ UseCase2 références: 2 cas d'utilisation trouvés
    - Authentification utilisateur
    - Consultation catalogue
    ✅ Relation 'Authentification utilisateur' détectée correctement
    ✅ Relation 'Consultation catalogue' détectée correctement

🔍 Test 3: Analyse des relations de 'Consultation catalogue'
  ✅ UseCase3 références: 2 cas d'utilisation trouvés
    - Authentification utilisateur
    - Gestion des commandes

🔄 Test 4: Vérification des relations bidirectionnelles
  ✅ Relation UseCase2 -> UseCase3 détectée
  ✅ Relation UseCase3 -> UseCase2 détectée
  📊 Résumé des relations:
   

In [18]:
def interprete_usecase_mock(usecases: List[Dict]) -> List[UseCase]:
    """
    Version simplifiée de interprete_usecase pour les tests
    """
    parsed_usecases = []
    
    for usecase_data in usecases:
        try:
            name = usecase_data.get('name', '').strip()
            action = determine_http_action(usecase_data)
            actors = extract_actors(usecase_data.get('actors', []))
            scenarios = extract_scenarios(usecase_data.get('scenarios', {}))
            preconditions = usecase_data.get('preconditions', [])
            postconditions = usecase_data.get('postconditions', [])
            
            # Génération simplifiée des composants
            dto_attributes = [
                DtoAttribute("private", "id", "Long", [Decorator("NotNull", "Required")]),
                DtoAttribute("private", "name", "String", [Decorator("NotBlank", "Required")])
            ]
            dto = Dto(f"{capitalize(name)}Dto", dto_attributes)
            
            resource_attributes = [Arg("private", attr.name, attr.type) for attr in dto_attributes]
            resource = Resource(f"{capitalize(name)}Resource", resource_attributes)
            
            service_methods = [
                Method(f"process{capitalize(name)}", [Arg("public", "dto", dto.name)], dto.name, "public")
            ]
            services = [Service(f"{capitalize(name)}Service", service_methods)]
            
            repo_methods = [
                Method("save", [Arg("public", "entity", f"{capitalize(name)}Entity")], f"{capitalize(name)}Entity", "public"),
                Method("findById", [Arg("public", "id", "Long")], f"Optional<{capitalize(name)}Entity>", "public")
            ]
            repositories = [Repository(f"{capitalize(name)}Entity", repo_methods)]
            
            usecase = UseCase(
                name=name,
                action=action,
                actors=actors,
                scenarios=scenarios,
                preconditions=preconditions,
                postconditions=postconditions,
                dto=dto,
                services=services,
                resource=resource,
                repositories=repositories
            )
            
            parsed_usecases.append(usecase)
            
        except Exception as e:
            print(f"Erreur lors du parsing de {usecase_data}: {e}")
            continue
    
    return parsed_usecases

In [19]:
def test_integration_complete():
    print("🧪 Test d'intégration complète")
    
    try:
        # Étape 1: Parser les cas d'utilisation
        print("  📝 Étape 1: Parsing des cas d'utilisation")
        parsed_usecases = interprete_usecase_mock(SAMPLE_USECASE_JSON)
        print(f"    ✅ {len(parsed_usecases)} cas d'utilisation parsés")
        
        # Mettre à jour la liste globale
        globals()['all_parsed_usecases'] = parsed_usecases
        
        # Étape 2: Analyser chaque cas d'utilisation
        print("  🔍 Étape 2: Analyse détaillée des cas d'utilisation")
        for i, uc in enumerate(parsed_usecases):
            print(f"    📋 Cas d'utilisation {i+1}: {uc.name}")
            print(f"      - Action HTTP: {uc.action}")
            print(f"      - Acteurs: {', '.join(uc.actors)}")
            print(f"      - Scénarios: {len(uc.scenarios.main)} principal(aux), {len(uc.scenarios.alternative)} alternatif(s)")
            print(f"      - DTO: {uc.dto.name} ({len(uc.dto.attributes)} attributs)")
            print(f"      - Services: {len(uc.services)} service(s)")
            print(f"      - Repositories: {len(uc.repositories)} repository(ies)")
            
            # Vérifier les attributs du DTO
            for attr in uc.dto.attributes:
                decorators_str = ', '.join([d.name for d in attr.decorators])
                print(f"        • {attr.name}: {attr.type} [{decorators_str}]")
        
        # Étape 3: Résoudre les relations
        print("  🔗 Étape 3: Résolution des relations entre cas d'utilisation")
        for uc in parsed_usecases:
            references = find_referenced_usecases(uc)
            if references:
                print(f"    📌 {uc.name} référence:")
                for ref in references:
                    print(f"      → {ref.name}")
            else:
                print(f"    📌 {uc.name}: aucune référence détectée")
        
        # Étape 4: Statistiques finales
        print("  📊 Étape 4: Statistiques finales")
        total_dtos = sum(len(uc.dto.attributes) for uc in parsed_usecases)
        total_services = sum(len(uc.services) for uc in parsed_usecases)
        total_repos = sum(len(uc.repositories) for uc in parsed_usecases)
        total_methods = sum(len(service.methods) for uc in parsed_usecases for service in uc.services)
        
        print(f"    ✅ Total DTOs attributs: {total_dtos}")
        print(f"    ✅ Total Services: {total_services}")
        print(f"    ✅ Total Repositories: {total_repos}")
        print(f"    ✅ Total Méthodes générées: {total_methods}")
        
        print("\n🎉 Test d'intégration complète RÉUSSI!")
        
    except Exception as e:
        print(f"  ❌ Erreur dans le test d'intégration: {e}")
        import traceback
        traceback.print_exc()

In [20]:
test_integration_complete()

🧪 Test d'intégration complète
  📝 Étape 1: Parsing des cas d'utilisation
Erreur lors du parsing de {'name': 'Authentification utilisateur', 'action': 'POST', 'actors': ['Utilisateur', 'Système'], 'scenarios': {'principal': ["L'utilisateur saisit ses identifiants", 'Le système vérifie les informations', "L'utilisateur accède au tableau de bord"], 'alternatif': ['Identifiants incorrects', "Le système affiche un message d'erreur"]}, 'preconditions': ["L'utilisateur possède un compte"], 'postconditions': ["L'utilisateur est connecté"], 'uses': '', 'extends': ''}: UseCase.__init__() missing 3 required positional arguments: 'uses', 'extends', and 'include'
Erreur lors du parsing de {'name': 'Gestion des commandes', 'action': 'POST', 'actors': ['Utilisateur'], 'scenarios': {'principal': ["L'utilisateur consulte le catalogue", "L'utilisateur ajoute des articles au panier", "L'utilisateur finalise la commande"], 'alternatif': ['Articles indisponibles', 'Le système propose des alternatives']}, '